In [1]:
import os
import re
import json
from copy import deepcopy
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
# from key import OPENAI_KEY, LANGSMITH_KEY    # Add your own keys
from key import DASHSCOPE_API_KEY, LANGSMITH_KEY     # Add your own keys
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# os.environ["OPENAI_API_KEY"] = OPENAI_KEY
os.environ["DASHSCOPE_API_KEY"] = DASHSCOPE_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_KEY

In [2]:
text_dir = "../data/text/Subtask_2_train.json"
anno = json.load(open(text_dir))

#### All Emotion Conversation Index
- Finding such conversations in training set
- Formatting them and getting explainations for emotion annotations
- Saving explainations
- Storing the conversations with all emotions in FAISS index 

In [4]:
### 保存索引。这个会话 conversation 具有所有情感的语句 utterance, 也就是说有7种 emotions 的 utterance。 

In [5]:
emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
idx = []
for i, a in enumerate(anno):
    e_flag = {e: False for e in emotions}
    for utt in a["conversation"]:
        e_flag[utt["emotion"]] = True
    if sum(e_flag.values()) == len(emotions):
        idx.append(i)

In [6]:
print("idx: ", idx)

idx:  [26, 101, 207, 222, 359, 370, 383, 572, 653, 733, 882, 1166]


In [7]:
def format_convo(idx, conversation, emotion=False):
    out = str(idx)
    # out = ""
    for i, utt in enumerate(conversation):
        out += f'\n{i+1}. {utt["speaker"]}: {utt["text"]}'
        if emotion:
            out += f' [{utt["emotion"]}]'
    return out

In [8]:
emotion_explaination_prompt = """
There are 6 basic emotions: Anger, Disgust, Fear, Joy, Sadness, Surprise. 
The emotion of the speaker is determined by the context of the conversation. 
If the emotion is not in any category, is a mix of several categories, or is ambiguous it can be categorized as "Neutral". 

Analyze the following conversation where emotion of each utterance is annotated in square brackets at the end. 
Give reasoning behind the annotation of each utterance.

{conversation}

Output a JSON in the following format:
[{{"utterance_ID": id,
  "text" : content,
  "speaker": speaker
  "emotion": emotion, 
  "explanation": detailed explanation}}
  ...
]
No plain text.
"""

In [9]:
# model = ChatOpenAI(openai_api_key=OPENAI_KEY)
model = ChatOpenAI(
                   # model="qwen-plus-2025-07-14", 
                   # model="qwen3.8-max-0902",
                   model="qwen3.8-flash",
                   openai_api_key=DASHSCOPE_API_KEY,
                   base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")
output_parser = StrOutputParser()

emotion_explaination_prompt = ChatPromptTemplate.from_template(emotion_explaination_prompt)
emotion_explaination_chain = emotion_explaination_prompt | model | output_parser

In [10]:
emo_convos = [format_convo(i, anno[i]["conversation"], True) for i in idx]
batch = [{"conversation": convo} for convo in emo_convos]

In [11]:
# outs = emotion_explaination_chain.batch(batch, config={"max_concurrency": 5})
outs = emotion_explaination_chain.batch(batch, config={"max_concurrency": 1})

In [12]:
print("outs: ", outs)

outs:  ['[\n  {\n    "utterance_ID": 1,\n    "text": "Ok , bye . Well , Monica not coming , it is just gonna be me and Rachel .",\n    "speaker": "Ross",\n    "emotion": "neutral",\n    "explanation": "Ross is simply reporting the change in plans and stating who will be present. The tone is informational and matter-of-fact, with no clear sign of anger, fear, joy, disgust, sadness, or surprise, so it is annotated as neutral."\n  },\n  {\n    "utterance_ID": 2,\n    "text": "Oh . Well , hold on camper , are you sure you have thought this thing through ?",\n    "speaker": "Chandler",\n    "emotion": "neutral",\n    "explanation": "Chandler is checking Ross\'s plan in a teasing but relatively calm way. Although there is mild playful concern, the utterance mainly functions as a question without a strong dominant basic emotion, so it is best categorized as neutral."\n  },\n  {\n    "utterance_ID": 3,\n    "text": "It is laundry . The thinking through is minimal .",\n    "speaker": "Ross",\n 

In [12]:
# Helper functions to fix a faulty json output from GPT
# def strip(s):
#     return re.sub('[^0-9a-zA-Z]+', '', s.strip())

# def fix_json(s):
#     emo_json = [] 
#     cur_dict = {}
#     for line in s.split("\n"):
#         print("line:",line)
#         components = line.strip().split(":")
#         print("components:",components)
#         key = strip(components[0])
#         print("key:",key)
#         if key == "utteranceID":
#             if len(cur_dict) != 0:
#                 emo_json.append(cur_dict)
#                 cur_dict = {}
#             cur_dict["utterance_ID"] = strip(components[1]) 
#         elif key == "text":
#             cur_dict[key] = ":".join(components[1:])
#         elif key == "speaker":
#             cur_dict[key] = strip(components[1])
#         elif key == "emotion":
#             cur_dict[key] = strip(components[1])
#         else:
#             cur_dict[key] = components[1].strip()[1:-3]
#     return emo_json

In [13]:
# Helper functions to fix a faulty json output from qwen-plus/qwen3.8-flash
def strip(s):
    return re.sub('[^0-9a-zA-Z]+', '', s.strip())

def fix_json(s):
    emo_json = [] 
    cur_dict = {}
    for line in s.split("\n"):
        print("line:", line)
        components = line.strip().split(":")
        print("components:", components)
        key = strip(components[0])
        print("key:", key)
        if key == "utteranceID":
            if len(cur_dict) != 0:
                emo_json.append(cur_dict)
                cur_dict = {}
            cur_dict["utterance_ID"] = strip(components[1]) 
        elif key == "text":
            # cur_dict[key] = ":".join(components[1:])
            cur_dict[key] = components[1].lstrip(' "').rstrip('",')
        elif key == "speaker":
            cur_dict[key] = strip(components[1])
        elif key == "emotion":
            cur_dict[key] = strip(components[1])
        elif key == "explanation":
            cur_dict[key] = components[1].lstrip(' "').rstrip('",')
        else:
            # cur_dict[key] = components[1].strip()[1:-3]
            pass
    return emo_json

In [14]:
emo_explain_dict = {}

for i, k in enumerate(idx):
    try:
        v = json.loads(outs[i])
    except:
        v = fix_json(outs[i])
    emo_explain_dict[k] = v

In [15]:
# json.dump(emo_explain_dict, open("emotion_explainations.json", "w"))
json.dump(emo_explain_dict, open("emotion_explainations1.json", "w"))

##### Creating FAISS Index

使用 OpenAI 的 Embeddings

In [65]:
# embeddings = OpenAIEmbeddings()
# all_emo_convos = [format_convo(i, anno[i]["conversation"], False) for i in idx]
# db = FAISS.from_texts(all_emo_convos, embeddings)
# db.save_local("all_emotion_index")

使用 Qwen / DashScope 时的写法（如果你不是用 OpenAI）

如果你用的是阿里云 DashScope 的 Qwen embeddings，那么就必须使用正确的类，不能用 OpenAIEmbeddings。

阿里云对应的是：

In [18]:
from langchain_community.embeddings import DashScopeEmbeddings
from tqdm import tqdm

embeddings = DashScopeEmbeddings(
    # model="text-embedding-v4",  # 或者你实际使用的模型
    model="qwen3.7-text-embedding-flash",
    dashscope_api_key=DASHSCOPE_API_KEY
)

all_emo_convos = [format_convo(i, anno[i]["conversation"], False) for i in idx]

print("all_emo_convos:", all_emo_convos)

# 你的全部文本
texts = all_emo_convos

# 分批处理
batch_size = 10
all_vecs = []
all_docs = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    vecs = embeddings.embed_documents(batch)
    all_vecs.extend(vecs)
    all_docs.extend(batch)

# 组合为 (text, embedding) 格式
pairs = list(zip(all_docs, all_vecs))

# 构建 FAISS
db = FAISS.from_embeddings( text_embeddings=pairs,
                            embedding=embeddings)

# 保存
# db.save_local("all_emotion_index")
db.save_local("all_emotion_index1")

all_emo_convos: ['26\n1. Ross: Ok , bye . Well , Monica not coming , it is just gonna be me and Rachel .\n2. Chandler: Oh . Well , hold on camper , are you sure you have thought this thing through ?\n3. Ross: It is laundry . The thinking through is minimal .\n4. Chandler: It is just you and Rachel , just the two of you ? This is a date . You are going on a date .\n5. Ross: Nuh ... uh .\n6. Chandler: Yuh ... huh .\n7. Ross: So what are you saying here ? I should shave again , pick up some wine , what ?\n8. Chandler: Well , you may wanna rethink the dirty underwear .\n9. Chandler: This is basically the first time she is gonna see your underwear ... you want it to be dirty ?\n10. Ross: No .\n11. Chandler: Oh , and uh , the fabric softener ?\n12. Ross: Ok , ok , now what is wrong with my Snuggles ?', '101\n1. Barry: Sweetheart , just gimme ... gimme another chance , okay , we will start all over again . We will go back to Aruba .\n2. Bernice: Dr . Farber , we have got a bit of an emergency

100%|█████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  2.21it/s]


#### Cause Windows Index 
- Create windows based on the position of emotional utterance (beg, mid, end)
- Create FAISS indices for three types of windows for each 6 emotions for RAG. (18 Indices in total)
- Save the Cause Windows in JSON file

In [19]:
def get_window_beg(convo: list, size:int = 3) -> list:
    return deepcopy(convo[:size])

def get_window_end(convo: list, size:int = 6) -> list:
    return deepcopy(convo[-size:])

def get_window_mid(convo:list, idx:int, prev_size:int = 5, next_size:int = 2) -> list:
    return deepcopy(convo[max(0, idx-prev_size) : (idx+1) + next_size])

✅ 正确做法：你需要自己构造 causes 列表

我们需要在 loop 之前，先把 emotion-cause_pairs 解析成一个 dict：

In [20]:
import json
from copy import deepcopy

# text_dir = "../data/text/Subtask_2_train.json"
# anno = json.load(open(text_dir))

# def get_window_beg(convo: list, size:int = 3) -> list:
#     return deepcopy(convo[:size])

# def get_window_end(convo: list, size:int = 6) -> list:
#     return deepcopy(convo[-size:])

# def get_window_mid(convo:list, idx:int, prev_size:int = 5, next_size:int = 2) -> list:
#     return deepcopy(convo[max(0, idx-prev_size) : (idx+1) + next_size])

emotions = ["anger", "joy", "sadness", "surprise", "disgust", "fear"]
index_dict = {emo: {"beg": [], "mid": [], "end": []} for emo in emotions}

for a in anno:
    # ---- 1) 建立 cause 映射 ----
    cause_map = {}   # key = utterance_ID  value = [cause_utterance_ID list]

    for emo_pair in a["emotion-cause_pairs"]:
        emo_id, cause_id = emo_pair   # "3_surprise" , "1"
        utt_id = int(emo_id.split("_")[0])     # 3
        cause_map.setdefault(utt_id, []).append(int(cause_id))

    # ---- 2) 遍历 conversation ----
    convo = a["conversation"]

    for i, utt in enumerate(convo):
        emo = utt["emotion"]
        if emo == "neutral":
            continue

        utt_id = utt["utterance_ID"]
        causes = cause_map.get(utt_id, [])   # 没有 cause 时返回 []

        video_prefix = utt["video_name"].split(".")[0]

        if i == 0:
            index_dict[emo]["beg"].append(
                (utt_id, get_window_beg(convo), causes, video_prefix)
            )
        elif i == len(convo) - 1:
            index_dict[emo]["end"].append(
                (utt_id, get_window_end(convo), causes, video_prefix)
            )
        else:
            index_dict[emo]["mid"].append(
                (utt_id, get_window_mid(convo, i), causes, video_prefix)
            )


In [91]:
# emotions = ["anger", "joy", "sadness", "surprise", "disgust", "fear"]
# index_dict = {emo: {"beg": [], "mid": [], "end": []} for emo in emotions}

# for a in anno:
#     for i, utt in enumerate(a["conversation"]):
#         if utt["emotion"] != "neutral":
#             if i == 0:
#                 index_dict[utt["emotion"]]["beg"].append((utt["utterance_ID"], 
#                                                           get_window_beg(a["conversation"]),
#                                                           utt["causes"],
#                                                           utt["video_name"].split(".")[0]))
#             elif i == len(a["conversation"]) - 1:
#                 index_dict[utt["emotion"]]["end"].append((utt["utterance_ID"], 
#                                                           get_window_end(a["conversation"]),
#                                                           utt["causes"],
#                                                           utt["video_name"].split(".")[0]))
#             else:
#                 index_dict[utt["emotion"]]["mid"].append((utt["utterance_ID"], 
#                                                           get_window_mid(a["conversation"], i),
#                                                           utt["causes"],
#                                                           utt["video_name"].split(".")[0]))

In [21]:
def format_window(window: tuple, label:bool = False) -> str:
    idx, window, causes, name = window
    utt_idx = idx
    emo = None
    out_str = ""
    if not label:
        out_str += f"{name}\n"
    for i, utt in enumerate(window):
        if idx == utt["utterance_ID"]:
            emo = utt["emotion"]
            out_str += f'{i+1}. {utt["speaker"]}: {utt["text"]}'
            if label: out_str += f' [{emo}]\n'
            else: out_str += '\n'
            utt_idx = i+1
        else:
            out_str += f'{i+1}. {utt["speaker"]}: {utt["text"]}\n'
    return out_str

✅ 解决方法

把 window_strings 按 batch_size=10 切块，然后逐批 embed，再手动构建 FAISS index。

LangChain 自带的 FAISS.from_texts() 会默认一次性 embed 全部文本（超过10就报错），
所以必须自己控制批量。

✔ 推荐修复代码（最简洁可用）

通用文本向量-v3  向量模型

模型Code  text-embedding-v3

模型介绍  向量模型

通用文本向量，是通义实验室基于LLM底座的多语言文本统一向量模型，面向全球多个主流语种，提供高水准的向量服务，帮助开发者将文本数据快速转换为高质量的向量数据。

In [22]:
from langchain_community.vectorstores import FAISS
import numpy as np

def build_faiss_with_batch(texts, embeddings, batch_size=10):
    
#     embeddings = DashScopeEmbeddings(
#     model="text-embedding-v4",  # 或者你实际使用的模型
#     dashscope_api_key=DASHSCOPE_API_KEY
# )
    
    all_embeds = []

    # 手动分 batch 调用 DashScope embeddings （限制 10 条）
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        vecs = embeddings.embed_documents(batch)  # 每批最多 10
        all_embeds.extend(vecs)
    # 转为 numpy 数组
    all_embeds = np.array(all_embeds)

     # ⚠️ 正确的 pair 格式：[(text, vector), ...]
    text_embedding_pairs = list(zip(texts, all_embeds))
    
    # 手动构建 FAISS
    return FAISS.from_embeddings(text_embeddings=text_embedding_pairs,
                                 embedding=embeddings)

然后你的原代码：

In [23]:
for emo in emotions:
    for pos in ["beg", "mid", "end"]:
        window_strings = [format_window(window) for window in index_dict[emo][pos]]
        db = build_faiss_with_batch(window_strings, embeddings)
        # db.save_local(f"cause_windows/{emo}/{pos}")
        db.save_local(f"cause_windows1/{emo}/{pos}")

In [ ]:
# for emo in emotions:
#     for pos in ["beg", "mid", "end"]:
#         window_strings = [format_window(window) for window in index_dict[emo][pos]]
#         db = FAISS.from_texts(window_strings, embeddings)
#         db.save_local(f"cause_windows/{emo}/{pos}")        

In [24]:
cause_windows = {}

for emo in emotions:
    for pos in ["beg", "mid", "end"]:
        for window in index_dict[emo][pos]:
            window_str = format_window(window, True)
            idx, wdw, cs, name = window
            cause_windows[name] = window_str

# json.dump(cause_windows, open("cause_windows.json", "w"))
json.dump(cause_windows, open("cause_windows1.json", "w"))

In [ ]:
### 创建具有全部情感语句的会话 FAISS 和对应的情感语句窗口 FAISS, 这些情感语句窗口根据6种情感和3种位置进行分类保存